In [ ]:
import torch, random, numpy as np
import os
import re
import time
import threading
import pandas as pd
import xml.etree.ElementTree as ET
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 0. Environment & Seed Initialization
# ==========================================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Available GPUs: {torch.cuda.device_count()}")

# ==========================================
# 🛡️ Physical-Level Validator & Fallback Function
# ==========================================
def clean_and_fix_svg(text):
    fallback_svg = "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 200 200' width='256' height='256'><rect width='200' height='200' fill='black'/></svg>"
    
    allowed_tags = {
        'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline', 
        'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask', 
        'linearGradient', 'radialGradient', 'stop', 'text', 'tspan', 
        'title', 'desc', 'style', 'pattern', 'marker', 'filter'
    }
    
    match = re.search(r"<svg.*?>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if not match:
        print("⚠️ Warning: Model failed to generate valid SVG tags, enabling fallback mechanism!")
        return fallback_svg
        
    svg_code = match.group(0)
    
    try:
        root = ET.fromstring(svg_code)
        for elem in root.iter():
            tag_name = elem.tag.split('}')[-1] 
            if tag_name not in allowed_tags:
                print(f"⚠️ Intercepted hallucinated fake tag: <{tag_name}>, fallback triggered!")
                return fallback_svg
        return svg_code
        
    except Exception as e:
        print(f"⚠️ Intercepted corrupted XML structure: {e}, fallback triggered!")
        return fallback_svg


# ==========================================
# 🚀 Minimalist Single Model Loading 
# ==========================================
merged_model_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'model.safetensors' in files and 'config.json' in files:
        merged_model_path = root
        break

if not merged_model_path:
    raise FileNotFoundError("❌ Merged model not found! Please check if the Dataset is mounted correctly.")

print(f"🎯 Successfully locked onto the real merged model path: {merged_model_path}")

# Load Tokenizers (Independent for each GPU to prevent Rust borrow conflicts)
tokenizer_0 = AutoTokenizer.from_pretrained(merged_model_path, local_files_only=True)
tokenizer_0.padding_side = "left"
if tokenizer_0.pad_token is None:
    tokenizer_0.pad_token = tokenizer_0.eos_token

tokenizer_1 = AutoTokenizer.from_pretrained(merged_model_path, local_files_only=True)
tokenizer_1.padding_side = "left"
if tokenizer_1.pad_token is None:
    tokenizer_1.pad_token = tokenizer_1.eos_token

# Get ChatML token ID (Calculated only once)
im_end_id = tokenizer_0.convert_tokens_to_ids("<|im_end|>")
if im_end_id is None:
    im_end_id = 151645
print(f"🔒 Locked EOS Token IDs: Native=[{tokenizer_0.eos_token_id}], ChatML=[{im_end_id}]")

# Load GPU 0
print("⏳ Loading GPU 0...")
model_0 = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"}, 
    local_files_only=True
)

# Load GPU 1
print("⏳ Loading GPU 1...")
model_1 = AutoModelForCausalLM.from_pretrained(
    merged_model_path,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:1"},  
    local_files_only=True
)

print("✅ Dual-GPUs are ready!")


# ==========================================
# 📝 Exam Preparation & Prompt Alignment
# ==========================================
TEST_PROMPTS_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "test.csv" in files:
        TEST_PROMPTS_PATH = os.path.join(root, "test.csv")
        break

SUBMISSION_PATH = "/kaggle/working/submission.csv"
test_df = pd.read_csv(TEST_PROMPTS_PATH)

SYSTEM_PROMPT = """You are an expert SVG code generator. Your task is to generate clean, strictly valid, and standalone SVG code based on the user's text description.

You MUST adhere to the following strict rules:
1. STRICT OUTPUT: Output ONLY the raw SVG code. No markdown formatting (no ```xml), no HTML wrappers, and no conversational text.
2. THE CANVAS RULE: Always use exactly <svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 200 200' width='256' height='256'>.
3. THE ADAPTIVE KISS PRINCIPLE: For basic shapes, you MUST use primitives (<rect>, <circle>, <ellipse>, <line>, <polygon>, <polyline>). For complex/natural objects, use optimized <path> elements. ANTI-HALLUCINATION: NEVER invent non-existent tags like <triangle>, <square>, <star>, <curve>, <arc>, <background>, or <layer>. Use valid SVG alternatives (e.g., <polygon>, <rect>, <path>, <g>).
4. DRAWING ORDER: Render elements from back to front.
5. STYLE & COLOR: Use direct presentation attributes ONLY (e.g., fill='black').
6. SECURITY & VALIDITY: Ensure all tags are properly closed. Use single quotes for all attributes."""

prompt_template = "<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"

def prepare_test_prompt(user_text):
    return prompt_template.format(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_text
    )

# ==========================================
# ⚙️ Batch Painter Workflow
# ==========================================
def worker_task_batched(df_chunk, model, tokenizer, device, worker_id, results_list, batch_size=8):
    print(f"👷 Painter {worker_id} starts drawing on {device}! Concurrency acceleration enabled (Batch Size={batch_size})")
    
    prompts = df_chunk['prompt'].tolist()
    ids = df_chunk['id'].tolist()
    start_time = time.time()
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        
        formatted_prompts = [prepare_test_prompt(p) for p in batch_prompts]
        inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1536,
                temperature=0.2,
                do_sample=True,
                top_p=0.9,
                eos_token_id=[tokenizer.eos_token_id, im_end_id], 
                pad_token_id=tokenizer.pad_token_id
            )
        
        input_length = inputs.input_ids.shape[1]
        
        for j, output in enumerate(outputs):
            generated_text = tokenizer.decode(output[input_length:], skip_special_tokens=False)
            generated_text = generated_text.replace('<|im_end|>', '').strip()
            generated_text = generated_text.replace('```xml', '').replace('```', '')
            
            final_svg = clean_and_fix_svg(generated_text)
            results_list.append({"id": batch_ids[j], "svg": final_svg})
            
        current_done = min(i + batch_size, len(prompts))
        if current_done % 8 == 0 or current_done == len(prompts):
            elapsed = time.time() - start_time
            print(f"  [Painter {worker_id}] Drawn {current_done}/{len(prompts)} | Elapsed: {elapsed:.1f} s")

# ==========================================
# 🚀 Dual-Core Concurrent Dispatch
# ==========================================
mid = len(test_df) // 2
df_0 = test_df.iloc[:mid]  
df_1 = test_df.iloc[mid:]  

results_0 = []
results_1 = []

t0 = time.time()
print("🏃 Full Power: Enabling Batch=8 + 1536 Tokens + Smart Dual-Core Truncation...")

thread_0 = threading.Thread(target=worker_task_batched, args=(df_0, model_0, tokenizer_0, "cuda:0", 0, results_0, 8))
thread_1 = threading.Thread(target=worker_task_batched, args=(df_1, model_1, tokenizer_1, "cuda:1", 1, results_1, 8))

thread_0.start()
thread_1.start()

thread_0.join()
thread_1.join()

final_results = results_0 + results_1
sub_df = pd.DataFrame(final_results)
sub_df = sub_df.sort_values(by="id")
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"\n🎉 Submission generated! Saved to: {SUBMISSION_PATH}")
print(f"⏱️ Extreme Concurrency Total Time: {elapsed_min:.2f} minutes")
